In [2]:
import math
import time
import pandas as pd

TOL = 1e-10
MAX_ITER = 200



def f1(x):
    return 2*x**4 + 4*x**3 + 3*x**2 - 10*x - 15

def df1(x):
    return 8*x**3 + 12*x**2 + 6*x - 10


def f2(x):
    return (x + 3) * (x + 1) * (x - 2)**3

def df2(x):
    A = (x + 3) * (x + 1)
    return (2*x + 4) * (x - 2)**3 + A * 3 * (x - 2)**2


def f3(x):
    return 5*x**3 + x**2 - math.exp(1 - 2*x) + math.cos(x) + 20

def df3(x):
    return 15*x**2 + 2*x + 2*math.exp(1 - 2*x) - math.sin(x)


def f4(x):
    return x * math.sin(x) + 4

def df4(x):
    return math.sin(x) + x * math.cos(x)


def f5(x):
    return (x - 3)**5 * math.log(x)

def df5(x):
    return 5*(x - 3)**4 * math.log(x) + (x - 3)**5 / x


def f6(x):
    return x**10 - 1

def df6(x):
    return 10 * x**9


def bissecao(f, a, b, tol=TOL, max_iter=MAX_ITER):
    fa = f(a)
    fb = f(b)

    if fa == 0:
        return a, 0, False
    if fb == 0:
        return b, 0, False
    if fa * fb > 0:
        return None, 0, True

    for it in range(1, max_iter + 1):
        c = (a + b) / 2
        fc = f(c)

        if abs(fc) < tol or abs(b - a) / 2 < tol:
            return c, it, False

        if fa * fc < 0:
            b = c
            fb = fc
        else:
            a = c
            fa = fc

    return c, max_iter, True


def falsa_posicao(f, a, b, tol=TOL, max_iter=MAX_ITER):
    fa = f(a)
    fb = f(b)

    if fa == 0:
        return a, 0, False
    if fb == 0:
        return b, 0, False
    if fa * fb > 0:
        return None, 0, True

    c = a
    for it in range(1, max_iter + 1):
        c = (a * fb - b * fa) / (fb - fa)
        fc = f(c)

        if abs(fc) < tol:
            return c, it, False

        if fa * fc < 0:
            b = c
            fb = fc
        else:
            a = c
            fa = fc

    return c, max_iter, True


def ponto_fixo(g, x0, tol=TOL, max_iter=MAX_ITER, residual=None):
    x = x0

    for it in range(1, max_iter + 1):
        xn = g(x)

        if not math.isfinite(xn):
            return x, it - 1, True

        if residual is not None and abs(residual(xn)) < tol:
            return xn, it, False

        if abs(xn - x) < tol:
            return xn, it, False

        x = xn

    return x, max_iter, True

g1 = lambda x: x - f1(x) / df1(x)
g2 = lambda x: x - f2(x) / df2(x)
g3 = lambda x: x - f3(x) / df3(x)
g4 = lambda x: math.pi + math.asin(4 / x)   # para a raiz em [1, 5]
g5 = lambda x: x - f5(x) / df5(x)
g6 = lambda x: x - f6(x) / df6(x)

problemas = [
    ("f1", f1, (0, 3), 3, g1),
    ("f2", f2, (0, 5), 5, g2),
    ("f3", f3, (-5, 5), -5, g3),
    ("f4", f4, (1, 5), 5, g4),
    ("f5", f5, (2, 5), 2, g5),
    ("f6", f6, (0.8, 1.2), 0.8, g6),
]

resultados = []

for nome, f, (a, b), x0, g in problemas:
    # Bisseção
    t0 = time.perf_counter()
    raiz_b, it_b, err_b = bissecao(f, a, b)
    tempo_b = time.perf_counter() - t0

    # Falsa posição
    t0 = time.perf_counter()
    raiz_fp, it_fp, err_fp = falsa_posicao(f, a, b)
    tempo_fp = time.perf_counter() - t0

    # Ponto fixo
    t0 = time.perf_counter()
    raiz_pf, it_pf, err_pf = ponto_fixo(g, x0, residual=f)
    tempo_pf = time.perf_counter() - t0

    resultados.append({
        "funcao": nome,
        "bissecao_raiz": raiz_b,
        "bissecao_it": it_b,
        "bissecao_erro": err_b,
        "bissecao_tempo_s": tempo_b,

        "falsa_posicao_raiz": raiz_fp,
        "falsa_posicao_it": it_fp,
        "falsa_posicao_erro": err_fp,
        "falsa_posicao_tempo_s": tempo_fp,

        "ponto_fixo_raiz": raiz_pf,
        "ponto_fixo_it": it_pf,
        "ponto_fixo_erro": err_pf,
        "ponto_fixo_tempo_s": tempo_pf,
    })

df = pd.DataFrame(resultados)

pd.set_option("display.float_format", lambda x: f"{x:.12f}")

print(df)

%timeit bissecao(f1, 0, 3)
%timeit falsa_posicao(f1, 0, 3)
%timeit ponto_fixo(g1, 3, residual=f1)

  funcao   bissecao_raiz  bissecao_it  bissecao_erro  bissecao_tempo_s  \
0     f1  1.492878708610           35          False    0.000063085000   
1     f2  2.000122070312           13          False    0.000019447000   
2     f3 -0.929560459845           37          False    0.000072267000   
3     f4  4.323239543708           34          False    0.000027243000   
4     f5  3.007812500000            7          False    0.000013220000   
5     f6  1.000000000000            1          False    0.000003882000   

   falsa_posicao_raiz  falsa_posicao_it  falsa_posicao_erro  \
0      1.492878708662                78               False   
1      1.714409136228               200                True   
2      1.568769261078               200                True   
3      4.323239543733                 9               False   
4      2.593909764205               200                True   
5      0.999999999991                50               False   

   falsa_posicao_tempo_s  ponto_fixo_ra